In [0]:
import time
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/ef_precios_medicamentos/datos/"
ARCHIVOS = {
    "real": "datos_listos.csv",
    "100k": "sinteticos_100k.csv",
    "500k": "sinteticos_500k.csv",
    "1M": "sinteticos_1m.csv",
}

In [0]:
def medir(accion, repeticiones=3):
    accion()  # calentamiento: NO se registra
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        accion()
        tiempos.append(time.perf_counter() - t0)
    return tiempos, sorted(tiempos)[len(tiempos) // 2]

In [0]:
resultados = []
for etiqueta, nombre in ARCHIVOS.items():
    ruta = BASE + nombre
    df = spark.read.csv(ruta, header=True, inferSchema=True)
    df = df.withColumnRenamed(df.columns[0], "codigo_producto")
    filas = df.count()
    pruebas = {
        "1_lectura_y_conteo": lambda: spark.read.csv(ruta, header=True, inferSchema=True).count(),
        "2_filtro_marca": lambda: df.filter(F.col("clasificacion") == "MARCA").count(),
        "3_groupBy_promedio": lambda: df.groupBy("clasificacion", "periodo").agg(F.avg("precio_mediana")).collect(),
        "4_groupBy_mediana": lambda: df.groupBy("clasificacion", "periodo").agg(F.median("precio_mediana")).collect(),
    }
    for op, accion in pruebas.items():
        tiempos, med = medir(accion)
        resultados.append((etiqueta, filas, op, round(tiempos[0], 2), round(tiempos[1], 2), round(tiempos[2], 2), round(med, 2)))
        print(etiqueta, filas, op, "| mediana (s):", round(med, 2))

real 92816 1_lectura_y_conteo | mediana (s): 1.53
real 92816 2_filtro_marca | mediana (s): 0.59
real 92816 3_groupBy_promedio | mediana (s): 0.58
real 92816 4_groupBy_mediana | mediana (s): 0.54
100k 100000 1_lectura_y_conteo | mediana (s): 1.21
100k 100000 2_filtro_marca | mediana (s): 0.43
100k 100000 3_groupBy_promedio | mediana (s): 0.46
100k 100000 4_groupBy_mediana | mediana (s): 0.45
500k 500000 1_lectura_y_conteo | mediana (s): 1.55
500k 500000 2_filtro_marca | mediana (s): 0.58
500k 500000 3_groupBy_promedio | mediana (s): 0.64
500k 500000 4_groupBy_mediana | mediana (s): 0.63
1M 1000000 1_lectura_y_conteo | mediana (s): 2.06
1M 1000000 2_filtro_marca | mediana (s): 0.69
1M 1000000 3_groupBy_promedio | mediana (s): 0.86
1M 1000000 4_groupBy_mediana | mediana (s): 0.91


In [0]:
res = spark.createDataFrame(resultados, ["escala", "filas", "operacion", "t1_s", "t2_s", "t3_s", "mediana_s"])
res.write.mode("overwrite").saveAsTable("workspace.ef_precios_medicamentos.exp_resultados")
display(res.orderBy("operacion", "filas"))

escala,filas,operacion,t1_s,t2_s,t3_s,mediana_s
real,92816,1_lectura_y_conteo,1.53,1.6,1.49,1.53
100k,100000,1_lectura_y_conteo,1.32,1.21,1.18,1.21
500k,500000,1_lectura_y_conteo,1.55,1.47,1.69,1.55
1M,1000000,1_lectura_y_conteo,1.94,2.06,2.14,2.06
real,92816,2_filtro_marca,0.6,0.52,0.59,0.59
100k,100000,2_filtro_marca,0.43,0.42,0.44,0.43
500k,500000,2_filtro_marca,0.56,0.58,0.6,0.58
1M,1000000,2_filtro_marca,0.64,0.69,0.71,0.69
real,92816,3_groupBy_promedio,0.54,0.58,0.69,0.58
100k,100000,3_groupBy_promedio,0.47,0.46,0.44,0.46


Databricks visualization. Run in Databricks to view.

**Nota sobre la figura:** este gráfico se regeneró sin el conjunto de datos reales (92 816 filas). Por ser el primero en procesarse, sus tiempos fueron algo mayores que los de 100 000 filas, lo que generaba una caída artificial al inicio de las líneas. La figura compara solo los tres conjuntos sintéticos; los resultados reales se conservan en la tabla `exp_resultados`.

In [0]:

display(res.filter("escala != 'real'").orderBy("operacion", "filas"))

escala,filas,operacion,t1_s,t2_s,t3_s,mediana_s
100k,100000,1_lectura_y_conteo,1.32,1.21,1.18,1.21
500k,500000,1_lectura_y_conteo,1.55,1.47,1.69,1.55
1M,1000000,1_lectura_y_conteo,1.94,2.06,2.14,2.06
100k,100000,2_filtro_marca,0.43,0.42,0.44,0.43
500k,500000,2_filtro_marca,0.56,0.58,0.6,0.58
1M,1000000,2_filtro_marca,0.64,0.69,0.71,0.69
100k,100000,3_groupBy_promedio,0.47,0.46,0.44,0.46
500k,500000,3_groupBy_promedio,0.73,0.64,0.6,0.64
1M,1000000,3_groupBy_promedio,0.89,0.86,0.77,0.86
100k,100000,4_groupBy_mediana,0.47,0.44,0.45,0.45


Databricks visualization. Run in Databricks to view.

In [0]:
df = spark.read.csv(BASE + "sinteticos_1m.csv", header=True, inferSchema=True)
df.groupBy("clasificacion", "periodo").agg(F.median("precio_mediana")).collect()

[Row(clasificacion='MARCA', periodo=datetime.date(2026, 6, 1), median(precio_mediana)=6.78615),
 Row(clasificacion='GENERICO', periodo=datetime.date(2026, 4, 1), median(precio_mediana)=2.6779),
 Row(clasificacion='GENERICO', periodo=datetime.date(2026, 2, 1), median(precio_mediana)=2.67935),
 Row(clasificacion='MARCA', periodo=datetime.date(2026, 5, 1), median(precio_mediana)=6.7252),
 Row(clasificacion='GENERICO', periodo=datetime.date(2026, 6, 1), median(precio_mediana)=2.6568),
 Row(clasificacion='MARCA', periodo=datetime.date(2026, 4, 1), median(precio_mediana)=6.7278),
 Row(clasificacion='MARCA', periodo=datetime.date(2026, 2, 1), median(precio_mediana)=6.73295),
 Row(clasificacion='MARCA', periodo=datetime.date(2026, 1, 1), median(precio_mediana)=6.64645),
 Row(clasificacion='GENERICO', periodo=datetime.date(2026, 3, 1), median(precio_mediana)=2.6909),
 Row(clasificacion='GENERICO', periodo=datetime.date(2026, 1, 1), median(precio_mediana)=2.6827),
 Row(clasificacion='MARCA', per

In [0]:
df = spark.read.csv(BASE + "sinteticos_100k.csv", header=True, inferSchema=True)
df.groupBy("clasificacion", "periodo").agg(F.median("precio_mediana")).collect()

[Row(clasificacion='MARCA', periodo=datetime.date(2026, 5, 1), median(precio_mediana)=6.6224),
 Row(clasificacion='MARCA', periodo=datetime.date(2026, 4, 1), median(precio_mediana)=6.6179),
 Row(clasificacion='MARCA', periodo=datetime.date(2026, 2, 1), median(precio_mediana)=6.624),
 Row(clasificacion='GENERICO', periodo=datetime.date(2026, 3, 1), median(precio_mediana)=2.6944),
 Row(clasificacion='MARCA', periodo=datetime.date(2026, 3, 1), median(precio_mediana)=6.673),
 Row(clasificacion='MARCA', periodo=datetime.date(2026, 6, 1), median(precio_mediana)=6.6875),
 Row(clasificacion='GENERICO', periodo=datetime.date(2026, 1, 1), median(precio_mediana)=2.66825),
 Row(clasificacion='GENERICO', periodo=datetime.date(2026, 2, 1), median(precio_mediana)=2.6581),
 Row(clasificacion='GENERICO', periodo=datetime.date(2026, 5, 1), median(precio_mediana)=2.6643999999999997),
 Row(clasificacion='GENERICO', periodo=datetime.date(2026, 6, 1), median(precio_mediana)=2.6332000000000004),
 Row(clasifi